# Notebook 01 — Split + Baselines (WBS 3.1, 3.2)

**Person 1 / Quyên.** Input: `curated.zip` (curated Parquet) trên Google Drive.
Output: splits + `split_stats.csv` + Movie Mean RMSE + Popularity Top-N artifact + `metrics_partial.csv`.

**Gates (PLAN_Person1.md B3.1/B3.2):**
- KILL-LEAKAGE: max(train.rating_ts) < min(val.rating_ts) < min(test.rating_ts) — PHẢI pass
- Split reproducible: deterministic cutoffs từ data (không seed cho temporal)
- KILL-METRIC: RMSE ngoài band [0.6, 1.1] ⟹ dừng, điều tra
- KILL-CONTRACT: artifact khớp `contracts/CONTRACTS.md` §3.1

Runtime: **Runtime → Change runtime type → High-RAM nếu có**. ~10-15 phút.

In [9]:
# Mount Drive + folders
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/movielens32m'
CURATED = f'{BASE}/curated'          # curated parquet (from curated.zip)
ARTIFACTS = f'{BASE}/artifacts'      # output serving artifacts
EVID = f'{BASE}/evidence'           # output evidence (copy back to repo evidence/)
CKPT = f'{BASE}/checkpoints'        # ALS checkpoint dir (MANDATORY — PLAN INV7)
for d in [BASE, CURATED, ARTIFACTS, EVID, CKPT]:
    os.makedirs(d, exist_ok=True)
print('folders ready:', os.listdir(BASE))

# Unzip curated parquet (one-time; skip if already extracted)
if not os.path.isdir(f'{CURATED}/curated_ratings') and os.path.exists(f'{BASE}/curated.zip'):
    !unzip -q -o '{BASE}/curated.zip' -d /content/tmpzip
    # zip was created as data/curated/... -> move into place
    !mkdir -p '{BASE}' && cp -r /content/tmpzip/data/curated '{BASE}/' && rm -rf /content/tmpzip
    print('curated parquet extracted')
assert os.path.isdir(f'{CURATED}/curated_ratings'), 'ERR: curated_ratings not found on Drive — upload curated.zip first (see README_COLAB_SETUP.md)'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
folders ready: ['curated.zip', 'curated', 'artifacts', 'evidence', 'checkpoints']


In [10]:
# Install Java + PySpark (driver memory set BEFORE JVM starts — PLAN Domain Guard)
import os
os.environ['PYSPARK_PYTHON'] = 'python3'

!dpkg -l | grep -q openjdk-11 || (apt-get update -qq && apt-get install -qq -y openjdk-11-jre-headless)
import glob as _g
_jvm = sorted(_g.glob('/usr/lib/jvm/java-11*'))
assert _jvm, 'ERR: openjdk-11 not installed — run !apt-get install -y openjdk-11-jre-headless and retry'
os.environ['JAVA_HOME'] = _jvm[0]
print('JAVA_HOME ->', os.environ['JAVA_HOME'])
!pip install -q pyspark==3.5.7

import pyspark
print('pyspark', pyspark.__version__)

JAVA_HOME -> /usr/lib/jvm/java-11-openjdk-amd64
pyspark 3.5.7


In [11]:
import pandas as pd
from pyspark.sql import SparkSession, functions as F
# Copy parquet from Drive FUSE to LOCAL disk first — Drive FUSE full-scans are slow/unstable
# (disconnect root cause #2: 4 full scans of 32M rows through FUSE)
import os, shutil
LOCAL = '/content/curated'
if not os.path.isdir(f'{LOCAL}/curated_ratings'):
    print('copying curated parquet Drive -> local disk (~500MB, 1-3 min)...')
    shutil.copytree(CURATED, LOCAL)
print('local copy ready')

spark = (SparkSession.builder
         .appName('movielens-split')
         .master('local[*]')
         .config('spark.driver.memory', '6g')   # 6g heap: ~6GB còn lại cho Python/OS trên Colab
         .config('spark.sql.shuffle.partitions', '32')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
spark.sparkContext.setCheckpointDir(CKPT)   # MANDATORY (ALS OOM guard)
ratings = spark.read.parquet(f'{LOCAL}/curated_ratings').cache()
n_total = ratings.count()
print(f'curated ratings: {n_total:,}')
assert n_total == 32000204, 'ERR2: count mismatch vs PRD'

local copy ready
curated ratings: 32,000,204


## B3.1 — Global Temporal Split (70/15/15)

Nghiên cứu nền tảng (PLAN §Evidence Appendix): temporal split = default field-wide
(SIGIR'23, TOIS'23, RecSys'20); random split để leakage. Cutoff theo thời gian toàn cục.
Caveat disclose trong MODEL_DESIGN: MovieLens timestamps không hoàn toàn tin cậy (TOIS'23).

In [12]:
# Cutoffs theo quantile thời gian (deterministic)
# BUG #11 FIX: relativeError=0.0 (exact) làm sketch GK giữ ~32M values -> JVM chết.
# relativeError=1e-4: chênh lệch cutoff tối đa ~0.01% — chấp nhận được, memory nhỏ.
q = ratings.approxQuantile('timestamp', [0.70, 0.85], 1e-4)
assert len(q) == 2, 'ERR: quantile computation failed'
cut_val, cut_test = q[0], q[1]
print(f'cutoff val:  {cut_val}\ncutoff test: {cut_test}')

print('building splits (3 filters + counts)...')
train = ratings.filter(F.col('timestamp') < cut_val).cache()
val   = ratings.filter((F.col('timestamp') >= cut_val) & (F.col('timestamp') < cut_test)).cache()
test  = ratings.filter(F.col('timestamp') >= cut_test).cache()
n_tr, n_va, n_te = train.count(), val.count(), test.count()
print(f'train={n_tr:,} ({100*n_tr/n_total:.1f}%) val={n_va:,} ({100*n_va/n_total:.1f}%) test={n_te:,} ({100*n_te/n_total:.1f}%)')

# Free heap ngay tại đây: ratings cache không còn cần (baselines chỉ dùng train/val/test)
ratings.unpersist(blocking=True)
print('ratings cache unpersisted')

cutoff val:  1476348398.0
cutoff test: 1573258563.0
building splits (3 filters + counts)...
train=22,399,368 (70.0%) val=4,798,976 (15.0%) test=4,801,860 (15.0%)
ratings cache unpersisted


In [13]:
# GATE KILL-LEAKAGE — phải PASS trước khi đi tiếp
max_tr = train.agg({'timestamp': 'max'}).first()[0]
min_va, min_te = val.agg({'timestamp': 'min'}).first()[0], test.agg({'timestamp': 'min'}).first()[0]
leak_free = (max_tr < min_va) and (min_va < min_te)
print(f'max(train)={max_tr} < min(val)={min_va} < min(test)={min_te} : {leak_free}')
assert leak_free, 'KILL-LEAKAGE: temporal split bị leakage — DỪNG, điều tra'

# Persist split metadata (reproducibility)
split_stats = pd.DataFrame({
    'n_total': [n_total], 'n_train': [n_tr], 'n_val': [n_va], 'n_test': [n_te],
    'cut_val': [str(cut_val)], 'cut_test': [str(cut_test)],
    'method': ['global_temporal_70_15_15'], 'leak_free': [leak_free]})
split_stats.to_csv(f'{EVID}/split_stats.csv', index=False)
print(split_stats.T)

# Free heap: ratings cache no longer needed after split caches are materialized

max(train)=1476348395 < min(val)=1476348398 < min(test)=1573258563 : True
                                  0
n_total                    32000204
n_train                    22399368
n_val                       4798976
n_test                      4801860
cut_val                1476348398.0
cut_test               1573258563.0
method     global_temporal_70_15_15
leak_free                      True


## B3.2a — Movie Mean Baseline (RMSE)
Prediction(movie) = mean train rating của phim; phim chưa có trong train ⟹ global mean (PRD/ARCH).

In [14]:
from pyspark.sql import functions as F
from pyspark.ml.evaluation import RegressionEvaluator

global_mean = train.agg(F.avg('rating')).first()[0]
movie_mean = train.groupBy('movieId').agg(F.avg('rating').alias('m_mean'))

def rmse_movie_mean(df, name):
    pred = (df.join(movie_mean, 'movieId', 'left')
            .withColumn('prediction', F.coalesce(F.col('m_mean'), F.lit(global_mean))))
    ev = RegressionEvaluator(metricName='rmse', labelCol='rating')
    r = ev.evaluate(pred)
    print(f'MovieMean RMSE on {name}: {r:.4f} (global fallback mean={global_mean:.4f})')
    return r

rmse_mm_val = rmse_movie_mean(val, 'val')
rmse_mm_test = rmse_movie_mean(test, 'test')

MovieMean RMSE on val: 1.0092 (global fallback mean=3.5287)
MovieMean RMSE on test: 0.9939 (global fallback mean=3.5287)


## B3.2b — Popularity Top-N (min-support grid + deterministic check)
Weighted score theo PRD: rating_count + average_rating với minimum support.
Grid {50, 100, 500} — chọn theo val (INV6: giá trị cuối phải justified bằng số đo).

In [15]:
MIN_SUPPORT_GRID = [50, 100, 500]
pop_stats = (train.groupBy('movieId')
             .agg(F.count('*').alias('support'), F.avg('rating').alias('avg_rating')))
movies_meta = spark.read.parquet(f'{CURATED}/curated_movies').select('movieId', 'title', 'genres')

results = {}
for ms in MIN_SUPPORT_GRID:
    top = (pop_stats.filter(F.col('support') >= ms)
           .join(movies_meta, 'movieId')
           .orderBy(F.desc('avg_rating'), F.desc('support'), F.asc('movieId'))  # deterministic total order
           .limit(10))
    results[ms] = [r['movieId'] for r in top.collect()]
    print(f'min_support={ms}: top5 movieIds = {results[ms][:5]}')

# Deterministic check: chạy lại 2 lần phải identical
ms_check = 100
t2 = (pop_stats.filter(F.col('support') >= ms_check).join(movies_meta, 'movieId')
      .orderBy(F.desc('avg_rating'), F.desc('support'), F.asc('movieId')).limit(10))
deterministic = [r['movieId'] for r in t2.collect()] == results[ms_check]
print(f'popularity deterministic (min_support={ms_check}): {deterministic}')

min_support=50: top5 movieIds = [159817, 318, 142115, 858, 50]
min_support=100: top5 movieIds = [159817, 318, 142115, 858, 50]
min_support=500: top5 movieIds = [318, 858, 50, 527, 1221]
popularity deterministic (min_support=100): True


In [16]:
# GATE KILL-METRIC — MovieMean RMSE phải trong band [0.6, 1.1] (band ALS well-tuned 0.7-0.9; MovieMean thường ~0.9-1.05)
BAND = (0.6, 1.1)
print(f'MovieMean test RMSE = {rmse_mm_test:.4f}, band = {BAND}')
assert BAND[0] <= rmse_mm_test <= BAND[1], f'KILL-METRIC: RMSE {rmse_mm_test:.4f} ngoài band — điều tra trước khi tiếp tục'

# Save popularity artifact theo CONTRACTS.md §3.1 (KILL-CONTRACT)
import json, datetime
ms_final = 100   # GIÁ TRỊ TUNABLE — justification: so sánh val ranking metric ở notebook 03
pop_rows = (pop_stats.filter(F.col('support') >= ms_final).join(movies_meta, 'movieId')
            .withColumn('score', F.col('avg_rating'))
            .orderBy(F.desc('score'), F.desc('support'), F.asc('movieId')).limit(10).collect())
doc = {'scope': 'global', 'modelVersion': 'v1.0.0',
       'generatedAt': datetime.datetime.utcnow().isoformat() + 'Z',
       'items': [{'movieId': r['movieId'], 'title': r['title'], 'genres': r['genres'],
                  'rank': i + 1, 'score': round(r['score'], 3), 'support': int(r['support'])}
                 for i, r in enumerate(pop_rows)]}
with open(f'{ARTIFACTS}/popular_movies.json', 'w') as f:
    json.dump(doc, f, indent=2)
pd.DataFrame({'model': ['MovieMean'], 'rmse_val': [rmse_mm_val], 'rmse_test': [rmse_mm_test]}).to_csv(f'{EVID}/metrics_partial.csv', index=False)
print('artifacts + metrics saved to Drive')

MovieMean test RMSE = 0.9939, band = (0.6, 1.1)
artifacts + metrics saved to Drive


/tmp/ipykernel_3563/388453518.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generatedAt': datetime.datetime.utcnow().isoformat() + 'Z',


## ✅ Notebook 01 DONE khi:
- KILL-LEAKAGE gate PASS (đã assert)
- RMSE trong band [0.6, 1.1] (đã assert)
- `split_stats.csv`, `metrics_partial.csv`, `popular_movies.json` tồn tại trên Drive
- Copy 3 file này + output cell về repo (`evidence/`, `artifacts/`) và cập nhật CHECKLIST + WORKLOG